# Train a small transformer in Google Colab

This notebook trains a decoder-only language model with plain PyTorch. It automatically uses a GPU when Colab provides one.

Before running it, choose **Runtime → Change runtime type → T4 GPU**. Then run each cell from top to bottom.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

@dataclass
class Config:
    context_length: int = 64
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1

class LanguageModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(VOCAB_SIZE, config.d_model)
        self.position_embedding = nn.Embedding(config.context_length, config.d_model)
        layer = nn.TransformerEncoderLayer(
            config.d_model, config.n_heads, 4 * config.d_model,
            config.dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, config.n_layers)
        self.norm = nn.LayerNorm(config.d_model)
        self.output = nn.Linear(config.d_model, VOCAB_SIZE, bias=False)
        self.output.weight = self.token_embedding.weight

    def forward(self, tokens):
        length = tokens.size(1)
        positions = torch.arange(length, device=tokens.device)
        hidden = self.token_embedding(tokens) + self.position_embedding(positions)
        mask = torch.triu(torch.ones(length, length, dtype=torch.bool, device=tokens.device), 1)
        hidden = self.transformer(hidden, mask=mask)
        return self.output(self.norm(hidden))

def encode(text):
    return torch.tensor(list(text.encode("utf-8")), dtype=torch.long)

def decode(tokens):
    return bytes(tokens.tolist()).decode("utf-8", errors="replace")

def get_batch(tokens, batch_size, context_length):
    if len(tokens) <= context_length:
        raise ValueError(f"Upload text longer than {context_length} bytes.")
    starts = torch.randint(len(tokens) - context_length, (batch_size,))
    x = torch.stack([tokens[i:i + context_length] for i in starts])
    y = torch.stack([tokens[i + 1:i + context_length + 1] for i in starts])
    return x.to(device), y.to(device)

def train(model, text, steps=500, batch_size=32, learning_rate=3e-4):
    tokens = encode(text)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    model.train()
    for step in range(1, steps + 1):
        x, y = get_batch(tokens, batch_size, model.config.context_length)
        logits = model(x)
        loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step == 1 or step % 50 == 0 or step == steps:
            print(f"step {step:>4}/{steps} | loss {loss.item():.4f}")

@torch.no_grad()
def generate(model, prompt, new_tokens=100, temperature=0.8):
    model.eval()
    tokens = encode(prompt).unsqueeze(0).to(device)
    for _ in range(new_tokens):
        context = tokens[:, -model.config.context_length:]
        logits = model(context)[:, -1]
        if temperature == 0:
            next_token = logits.argmax(-1, keepdim=True)
        else:
            probabilities = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probabilities, 1)
        tokens = torch.cat((tokens, next_token), dim=1)
    return decode(tokens[0].cpu())

def save_model(model, path):
    torch.save({"config": asdict(model.config), "model_state": model.state_dict()}, path)

def load_model(path):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    model = LanguageModel(Config(**checkpoint["config"])).to(device)
    model.load_state_dict(checkpoint["model_state"])
    return model

## Upload training text and train

Upload one UTF-8 `.txt` file. For a quick test, a few pages of text is enough. Increase `steps` for a better result.

In [ ]:
from google.colab import files

uploaded = files.upload()
data_file = next(iter(uploaded))
text = uploaded[data_file].decode("utf-8")

torch.manual_seed(42)
model = LanguageModel(Config()).to(device)
train(model, text, steps=500)
save_model(model, "simple_transformer.pt")

## Generate text and download the checkpoint

In [ ]:
print(generate(model, "The ", new_tokens=200))
files.download("simple_transformer.pt")

## Optional: fine-tune on new text

This continues training the current model with a lower learning rate. Run it before the Colab session ends, or upload a saved checkpoint and call `load_model` first.

In [ ]:
uploaded = files.upload()
fine_tune_file = next(iter(uploaded))
fine_tune_text = uploaded[fine_tune_file].decode("utf-8")

train(model, fine_tune_text, steps=200, learning_rate=3e-5)
save_model(model, "fine_tuned_transformer.pt")
print(generate(model, "The ", new_tokens=200))
files.download("fine_tuned_transformer.pt")